# Deadtime explorer (channel threshold)

Interactive views of the double-pulse deadtime scans with channel-rate gating.

In [1]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from ipywidgets import interact, Dropdown, Checkbox
from IPython.display import clear_output
import sys

sns.set_context('talk')

SELECT = "(select)"

ROOT = Path.cwd().resolve().parent
sys.path.append(str(ROOT / 'src'))
from deadtime_analysis import DeadtimeAnalysis

DATA_FILES = [
    ROOT / 'data' / 'double_pulse_deadtime-12-22-25.jsonl',
]

EST_PATH = ROOT / 'data' / 'estimated_deadtime_10hz-12-22-25_full_channels.json'
BASE_EST_PATH = ROOT / 'data' / 'estimated_deadtime_10hz-12-22-25.json'

analysis_full = DeadtimeAnalysis.from_jsonl([str(p) for p in DATA_FILES])

with EST_PATH.open() as fh:
    estimates = json.load(fh)
est_df = pd.DataFrame(estimates)

base_est_df = None
if BASE_EST_PATH.exists():
    with BASE_EST_PATH.open() as fh:
        base_estimates = json.load(fh)
    base_est_df = pd.DataFrame(base_estimates)

print('Estimates rows:', len(est_df))
print('Data rows:', len(analysis_full.df))
print('Available pulse rates:', sorted(analysis_full.df['pulse_rate_hz'].dropna().unique()))


Estimates rows: 91
Data rows: 1638
Available pulse rates: [np.float64(10.0)]


## Interactive rate vs separation


In [2]:
pulse_rate_options = [SELECT] + sorted(analysis_full.df['pulse_rate_hz'].dropna().unique())
pulse_options = [SELECT] + sorted(analysis_full.df['num_pulses'].dropna().unique())
channel_options = ['(all)'] + sorted(analysis_full.df['channel_count'].dropna().unique())
window_options = ['(all)'] + sorted(analysis_full.df['windows'].dropna().unique())

y_min = None
y_max = None

@interact(
    pulse_rate=Dropdown(options=pulse_rate_options, value=SELECT, description='Pulse Rate (Hz)'),
    num_pulses=Dropdown(options=pulse_options, value=SELECT, description='Pulses'),
    channels=Dropdown(options=channel_options, value='(all)', description='Channels'),
    windows=Dropdown(options=window_options, value='(all)', description='Windows'),
    show_stars=Checkbox(value=True, description='Show stars'),
    show_vertical_bounds=Checkbox(value=True, description='Show vertical bounds'),
    show_notes=Checkbox(value=True, description='Show notes'),
    show_deadtime_estimate=Checkbox(value=True, description='Show deadtime estimate'),
)
def _plot_rates(pulse_rate, num_pulses, channels, windows, show_stars, show_vertical_bounds, show_notes, show_deadtime_estimate):
    clear_output(wait=True)
    if pulse_rate in (None, SELECT) or num_pulses in (None, SELECT):
        return

    df = analysis_full.df[analysis_full.df['pulse_rate_hz'] == pulse_rate].copy()
    df = df[df['num_pulses'] == num_pulses]

    ch_val = None if channels == '(all)' else channels
    win_val = None if windows == '(all)' else windows
    if ch_val is not None:
        df = df[df['channel_count'] == ch_val]
    if win_val is not None:
        df = df[df['windows'] == win_val]
    if df.empty:
        print('No data for selection')
        return

    match = None
    est_match = est_df[(est_df['pulse_rate_hz'] == pulse_rate) & (est_df['num_pulses'] == num_pulses)]
    if ch_val is not None:
        est_match = est_match[est_match['channel_count'] == ch_val]
    if win_val is not None:
        est_match = est_match[est_match['windows'] == win_val]
    if not est_match.empty:
        match = est_match.iloc[0]

    base_match = None
    if base_est_df is not None:
        base_sel = base_est_df[(base_est_df['pulse_rate_hz'] == pulse_rate) & (base_est_df['num_pulses'] == num_pulses)]
        if ch_val is not None:
            base_sel = base_sel[base_sel['channel_count'] == ch_val]
        if win_val is not None:
            base_sel = base_sel[base_sel['windows'] == win_val]
        if not base_sel.empty:
            base_match = base_sel.iloc[0]

    notes = []
    deadtime_range_ns = None
    deadtime_estimate_ns = None
    highlights = None

    reference_match = base_match if base_match is not None else match
    if reference_match is not None:
        lb = reference_match.get('min_all_pulses_lower_bound_ns')
        resp = reference_match.get('min_all_pulses_response_ns')
        if lb is not None and resp is not None:
            highlights = [lb, resp]
            deadtime_range_ns = (lb, resp)
            notes.append(f'Rate-doubling range: {lb:.0f}-{resp:.0f} ns')

    if match is not None:
        estimate = match.get('min_all_pulses_response_ns')
        if estimate is not None:
            deadtime_estimate_ns = estimate
            notes.append(f'Full-channel estimate: {estimate:.0f} ns')

    deadtime_range_text = '\n'.join(notes) if notes else None

    ana = DeadtimeAnalysis(df, single_factor=analysis_full.single_factor, double_factor=analysis_full.double_factor)
    ana.plot_rate_vs_separation_by_channels(
        pulse_rate,
        num_pulses=num_pulses,
        highlight_separations_ns=highlights,
        show_stars=show_stars,
        show_vertical_bounds=show_vertical_bounds,
        show_notes=show_notes,
        deadtime_range_ns=deadtime_range_ns,
        deadtime_range_text=deadtime_range_text,
        deadtime_estimate_ns=deadtime_estimate_ns if show_deadtime_estimate else None,
        y_min=y_min,
        y_max=y_max,
    )
    ana.plot_rate_vs_separation_by_windows(
        pulse_rate,
        num_pulses=num_pulses,
        highlight_separations_ns=highlights,
        show_stars=show_stars,
        show_vertical_bounds=show_vertical_bounds,
        show_notes=show_notes,
        deadtime_range_ns=deadtime_range_ns,
        deadtime_range_text=deadtime_range_text,
        deadtime_estimate_ns=deadtime_estimate_ns if show_deadtime_estimate else None,
        y_min=y_min,
        y_max=y_max,
    )


interactive(children=(Dropdown(description='Pulse Rate (Hz)', options=('(select)', np.float64(10.0)), value='(…

## Interactive channels/event vs separation


In [3]:
@interact(
    pulse_rate=Dropdown(options=pulse_rate_options, value=SELECT, description='Pulse Rate (Hz)'),
    num_pulses=Dropdown(options=pulse_options, value=SELECT, description='Pulses'),
    channels=Dropdown(options=channel_options, value='(all)', description='Channels'),
    windows=Dropdown(options=window_options, value='(all)', description='Windows'),
    show_expected=Checkbox(value=True, description='Show expected'),
    show_threshold=Checkbox(value=True, description='Show threshold'),
)
def _plot_channels(pulse_rate, num_pulses, channels, windows, show_expected, show_threshold):
    clear_output(wait=True)
    if pulse_rate in (None, SELECT) or num_pulses in (None, SELECT):
        return

    df = analysis_full.df[analysis_full.df['pulse_rate_hz'] == pulse_rate].copy()
    df = df[df['num_pulses'] == num_pulses]

    ch_val = None if channels == '(all)' else channels
    win_val = None if windows == '(all)' else windows
    if ch_val is not None:
        df = df[df['channel_count'] == ch_val]
    if win_val is not None:
        df = df[df['windows'] == win_val]
    if df.empty:
        print('No data for selection')
        return

    ana = DeadtimeAnalysis(df, single_factor=analysis_full.single_factor, double_factor=analysis_full.double_factor)
    ana.plot_channels_per_event_vs_separation_by_channels(
        pulse_rate,
        num_pulses=num_pulses,
        show_expected=show_expected,
        show_threshold=show_threshold,
    )
    ana.plot_channels_per_event_vs_separation_by_windows(
        pulse_rate,
        num_pulses=num_pulses,
        show_expected=show_expected,
        show_threshold=show_threshold,
    )


interactive(children=(Dropdown(description='Pulse Rate (Hz)', options=('(select)', np.float64(10.0)), value='(…

## Heatmap of min multi-pulse separation


In [4]:
pulse_rate_heatmap = [SELECT] + sorted(est_df['pulse_rate_hz'].dropna().unique())
pulse_count_heatmap = [SELECT] + sorted(est_df['num_pulses'].dropna().unique())

@interact(
    pulse_rate=Dropdown(options=pulse_rate_heatmap, value=SELECT, description='Pulse Rate (Hz)'),
    num_pulses=Dropdown(options=pulse_count_heatmap, value=SELECT, description='Pulses'),
)
def _plot_heatmap(pulse_rate, num_pulses):
    clear_output(wait=True)
    if pulse_rate in (None, SELECT) or num_pulses in (None, SELECT):
        return
    sub = est_df[(est_df['pulse_rate_hz'] == pulse_rate) & (est_df['num_pulses'] == num_pulses)]
    if sub.empty:
        print('No estimates for selection')
        return
    pivot = sub.pivot_table(index='channel_count', columns='windows', values='min_all_pulses_response_us')
    plt.figure(figsize=(10, 6))
    sns.heatmap(pivot, annot=True, fmt='.2f', cmap='viridis')
    plt.title(f'Min multi-pulse separation (us, full channels) @ {pulse_rate:.0f} Hz, pulses={int(num_pulses)}')
    plt.xlabel('Windows')
    plt.ylabel('Channels')
    plt.tight_layout()
    plt.show()


interactive(children=(Dropdown(description='Pulse Rate (Hz)', options=('(select)', np.float64(10.0)), value='(…